<a href="https://colab.research.google.com/github/Anirban511/RNA_LLM_Cellstate_prediction/blob/main/Copy_of_Cell_State_Prediction_RNA_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Install:
#pip install torch torchvision torchaudio
#pip install torch-geometric

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from ..modules import (
    TransformerLayer,
    LearnedPositionalEmbedding,
    SinusoidalPositionalEmbedding,
    RobertaLMHead,
    ESM1bLayerNorm,
    ContactPredictionHead,
)


class BioBertModel(nn.Module):   # ProteinBertModel(nn.Module):
    @classmethod
    def add_args(cls, parser):
        # existing model architecture args (keep original names if present)
        parser.add_argument(
            "--num_layers", default=36, type=int, metavar="N", help="number of layers"
        )
        parser.add_argument(
            "--embed_dim", default=1280, type=int, metavar="N", help="embedding dimension"
        )
        parser.add_argument(
            "--logit_bias", action="store_true", help="whether to apply bias to logits"
        )
        parser.add_argument(
            "--ffn_embed_dim",
            default=5120,
            type=int,
            metavar="N",
            help="embedding dimension for FFN",
        )
        parser.add_argument(
            "--attention_heads",
            default=20,
            type=int,
            metavar="N",
            help="number of attention heads",
        )

        # new args for classification head and fine-tuning behavior
        parser.add_argument(
            "--num_cell_states",
            default=2,
            type=int,
            help="number of classes for cell-state classification (e.g., pre-tumor, normal)",
        )
        parser.add_argument(
            "--classifier_hidden_dim",
            default=512,
            type=int,
            help="hidden size for cell-state MLP head",
        )
        parser.add_argument(
            "--classifier_dropout", default=0.1, type=float, help="dropout for classification head"
        )
        parser.add_argument(
            "--pooling",
            default=None,
            choices=["cls", "mean"],
            help="how to pool sequence for classification; default chooses 'cls' if model prepends BOS/CLS else 'mean'",
        )
        parser.add_argument(
            "--freeze_encoder", action="store_true", help="freeze encoder weights at start of fine-tuning"
        )
        parser.add_argument(
            "--unfreeze_last_n",
            default=0,
            type=int,
            help="if freezing encoder, optionally unfreeze last N transformer layers",
        )
        parser.add_argument(
            "--multitask_mlm_weight",
            default=0.0,
            type=float,
            help="weight for MLM loss when doing multi-task (classification + MLM). 0 => no MLM loss added",
        )

    def __init__(self, args, alphabet):
        super().__init__()
        self.args = args

        # compatibility: some codebases use args.num_layers vs args.layers
        if not hasattr(self.args, "layers") and hasattr(self.args, "num_layers"):
            self.args.layers = self.args.num_layers

        # basic alphabet and special tokens
        self.alphabet_size = len(alphabet)
        self.padding_idx = alphabet.padding_idx
        self.mask_idx = alphabet.mask_idx
        self.cls_idx = alphabet.cls_idx
        self.eos_idx = alphabet.eos_idx
        self.prepend_bos = alphabet.prepend_bos
        self.append_eos = alphabet.append_eos

        # choose pooling behavior if not passed explicitly
        self.pooling = getattr(self.args, "pooling", None)
        if self.pooling is None:
            self.pooling = "cls" if self.prepend_bos else "mean"

        # keep backward-compatible emb_layer_norm_before flag
        self.emb_layer_norm_before = getattr(self.args, "emb_layer_norm_before", False)

        if self.args.arch == "roberta_large":
            self.model_version = "ESM-1b"
            self._init_submodules_esm1b()
        else:
            self.model_version = "ESM-1"
            self._init_submodules_esm1()

        # optionally freeze encoder right away (useful for quick head-only training)
        if getattr(self.args, "freeze_encoder", False):
            self.set_encoder_trainable(trainable=False, unfreeze_last_n=getattr(self.args, "unfreeze_last_n", 0))

    def _init_submodules_common(self):
        # token embeddings
        self.embed_tokens = nn.Embedding(
            self.alphabet_size, self.args.embed_dim, padding_idx=self.padding_idx
        )

        # transformer layers
        self.layers = nn.ModuleList(
            [
                TransformerLayer(
                    self.args.embed_dim,
                    self.args.ffn_embed_dim,
                    self.args.attention_heads,
                    add_bias_kv=(self.model_version != "ESM-1b"),
                    use_esm1b_layer_norm=(self.model_version == "ESM-1b"),
                )
                for _ in range(self.args.layers)
            ]
        )

        # contact head (keeps original behavior)
        self.contact_head = ContactPredictionHead(
            self.args.layers * self.args.attention_heads,
            self.prepend_bos,
            self.append_eos,
            eos_idx=self.eos_idx,
        )

        # classification head for cell-state prediction
        # small MLP: embed_dim -> hidden -> num_cell_states
        self.cell_state_head = nn.Sequential(
            nn.Linear(self.args.embed_dim, getattr(self.args, "classifier_hidden_dim", 512)),
            nn.ReLU(inplace=True),
            nn.Dropout(getattr(self.args, "classifier_dropout", 0.1)),
            nn.Linear(getattr(self.args, "classifier_hidden_dim", 512), getattr(self.args, "num_cell_states", 2)),
        )

    def _init_submodules_esm1b(self):
        self._init_submodules_common()
        self.embed_scale = 1
        self.embed_positions = LearnedPositionalEmbedding(
            self.args.max_positions, self.args.embed_dim, self.padding_idx
        )
        self.emb_layer_norm_before = (
            ESM1bLayerNorm(self.args.embed_dim) if self.emb_layer_norm_before else None
        )
        self.emb_layer_norm_after = ESM1bLayerNorm(self.args.embed_dim)
        self.lm_head = RobertaLMHead(
            embed_dim=self.args.embed_dim,
            output_dim=self.alphabet_size,
            weight=self.embed_tokens.weight,
        )

    def _init_submodules_esm1(self):
        self._init_submodules_common()
        self.embed_scale = math.sqrt(self.args.embed_dim)
        self.embed_positions = SinusoidalPositionalEmbedding(self.args.embed_dim, self.padding_idx)
        self.embed_out = nn.Parameter(torch.zeros((self.alphabet_size, self.args.embed_dim)))
        self.embed_out_bias = None
        if self.args.final_bias:
            self.embed_out_bias = nn.Parameter(torch.zeros(self.alphabet_size))

    def forward(self, tokens, repr_layers=[], need_head_weights=False, return_contacts=False, masked_tokens=None):
        """
        Forward returns a dict:
        - 'logits': token logits (B, T, V) (MLM logits or linear->vocab)
        - 'representations': dict of requested layer representations
        - 'attentions' (optional)
        - 'contacts' (optional)
        - 'cell_state_logits' : classification logits for the whole sequence (B, num_cell_states)
        """
        if return_contacts:
            need_head_weights = True

        assert tokens.ndim == 2
        padding_mask = tokens.eq(self.padding_idx)  # B, T

        x = self.embed_scale * self.embed_tokens(tokens)

        if getattr(self.args, "token_dropout", False):
            x.masked_fill_((tokens == self.mask_idx).unsqueeze(-1), 0.0)
            # x: B x T x C
            mask_ratio_train = 0.15 * 0.8
            src_lengths = (~padding_mask).sum(-1)
            mask_ratio_observed = (tokens == self.mask_idx).sum(-1).float() / src_lengths
            x = x * (1 - mask_ratio_train) / (1 - mask_ratio_observed)[:, None, None]

        x = x + self.embed_positions(tokens)

        if self.model_version == "ESM-1b":
            if self.emb_layer_norm_before:
                x = self.emb_layer_norm_before(x)
            if padding_mask is not None:
                x = x * (1 - padding_mask.unsqueeze(-1).type_as(x))

        repr_layers = set(repr_layers)
        hidden_representations = {}
        if 0 in repr_layers:
            hidden_representations[0] = x

        if need_head_weights:
            attn_weights = []

        # (B, T, E) => (T, B, E)
        x = x.transpose(0, 1)

        if not padding_mask.any():
            padding_mask = None

        for layer_idx, layer in enumerate(self.layers):
            x, attn = layer(
                x, self_attn_padding_mask=padding_mask, need_head_weights=need_head_weights
            )
            if (layer_idx + 1) in repr_layers:
                hidden_representations[layer_idx + 1] = x.transpose(0, 1)
            if need_head_weights:
                # (H, B, T, T) => (B, H, T, T)
                attn_weights.append(attn.transpose(1, 0))

        # at this point `x` is (T, B, E)
        # take a copy as features (before mapping to token logits) and transpose to (B, T, E)
        if self.model_version == "ESM-1b":
            x = self.emb_layer_norm_after(x)
            features = x.transpose(0, 1)  # (B, T, E)
            # last hidden representation should have layer norm applied
            if (layer_idx + 1) in repr_layers:
                hidden_representations[layer_idx + 1] = features
            # produce token logits via LM head (unchanged API)
            logits = self.lm_head(features, masked_tokens)
        else:
            # for ESM-1, keep a features copy before the final linear->vocab mapping
            features = x.transpose(0, 1)  # (B, T, E)
            # now map to vocab logits
            token_logits = F.linear(x, self.embed_out, bias=self.embed_out_bias)  # (T, B, V)
            logits = token_logits.transpose(0, 1)  # (B, T, V)

        # ---- classification pooling ----
        # features: (B, T, E)
        if self.pooling == "cls":
            # if model was trained with prepend_bos/cls, use position 0
            if self.prepend_bos:
                pooled = features[:, 0, :]  # (B, E)
            else:
                # fallback: masked mean
                if padding_mask is None:
                    pooled = features.mean(dim=1)
                else:
                    mask = (~padding_mask).unsqueeze(-1).type_as(features)  # (B, T, 1)
                    summed = (features * mask).sum(dim=1)
                    lengths = mask.sum(dim=1).clamp(min=1.0)
                    pooled = summed / lengths
        else:  # mean pooling
            if padding_mask is None:
                pooled = features.mean(dim=1)
            else:
                mask = (~padding_mask).unsqueeze(-1).type_as(features)
                summed = (features * mask).sum(dim=1)
                lengths = mask.sum(dim=1).clamp(min=1.0)
                pooled = summed / lengths

        cell_state_logits = self.cell_state_head(pooled)  # (B, num_cell_states)

        result = {"logits": logits, "representations": hidden_representations, "cell_state_logits": cell_state_logits}
        if need_head_weights:
            # attentions: B x L x H x T x T
            attentions = torch.stack(attn_weights, 1)
            if self.model_version == "ESM-1":
                # ESM-1 models have an additional null-token for attention, which we remove
                attentions = attentions[..., :-1]
            if padding_mask is not None:
                attention_mask = 1 - padding_mask.type_as(attentions)
                attention_mask = attention_mask.unsqueeze(1) * attention_mask.unsqueeze(2)
                attentions = attentions * attention_mask[:, None, None, :, :]
            result["attentions"] = attentions
            if return_contacts:
                contacts = self.contact_head(tokens, attentions)
                result["contacts"] = contacts

        return result

    def predict_contacts(self, tokens):
        return self(tokens, return_contacts=True)["contacts"]

    def predict_cell_state(self, tokens, return_probs=False):
        """
        Convenience wrapper: returns argmax class or probabilities.
        """
        self.eval()
        with torch.no_grad():
            out = self(tokens)
            logits = out["cell_state_logits"]  # (B, C)
            if return_probs:
                probs = F.softmax(logits, dim=-1)
                return probs
            else:
                return logits.argmax(dim=-1)

    def set_encoder_trainable(self, trainable=True, unfreeze_last_n=0):
        """
        Set encoder (embeddings + transformer layers + positional) trainability.
        If unfreeze_last_n > 0, last N transformer layers will be set to trainable regardless.
        """
        # embeddings and positions
        if hasattr(self, "embed_tokens"):
            for p in self.embed_tokens.parameters():
                p.requires_grad = trainable
        if hasattr(self, "embed_positions"):
            try:
                for p in self.embed_positions.parameters():
                    p.requires_grad = trainable
            except Exception:
                pass

        # all transformer layers
        for p in self.layers.parameters():
            p.requires_grad = trainable

        # optionally unfreeze last N layers
        if unfreeze_last_n > 0:
            for layer in list(self.layers)[-unfreeze_last_n:]:
                for p in layer.parameters():
                    p.requires_grad = True

    def get_optimizer_param_groups(self, base_lr, head_lr, weight_decay=0.0):
        """
        Return parameter groups for optimizers:
         - encoder params use base_lr
         - classification head params use head_lr
        Use this to pass directly to an optimizer (e.g., AdamW).
        """
        encoder_params = []
        head_params = []
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            # classify by name: cell_state_head params go to head_lr
            if "cell_state_head" in name or "cell_state" in name:
                head_params.append(param)
            else:
                encoder_params.append(param)
        groups = [
            {"params": encoder_params, "lr": base_lr, "weight_decay": weight_decay},
            {"params": head_params, "lr": head_lr, "weight_decay": weight_decay},
        ]
        return groups

    @property
    def num_layers(self):
        return self.args.layers

# Suppose you have a batch of tokens and labels
outputs = model(tokens, return_cell_states=True)
logits = outputs["cell_state_logits"]   # (batch_size, num_classes)

batch_acc = accuracy_fn(logits, labels)
print("Batch accuracy:", batch_acc)

all_logits, all_labels = [], []
with torch.no_grad():
    for batch_tokens, batch_labels in test_loader:
        outputs = model(batch_tokens, return_cell_states=True)
        logits = outputs["cell_state_logits"]
        acc = accuracy_fn(logits, batch_labels)
        print("Batch acc:", acc)

In [ ]:
class MultiModalCellStateModel(nn.Module):
    def __init__(self, base_model, cell_feature_dim, num_cell_states=3):
        super().__init__()
        self.base_model = base_model
        self.cell_feature_dim = cell_feature_dim
        self.num_cell_states = num_cell_states

        # small MLP for cell-level numeric features
        self.cell_feature_encoder = nn.Sequential(
            nn.Linear(cell_feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        # combine embeddings from sequence + cell features
        self.fusion_head = nn.Sequential(
            nn.Linear(base_model.args.embed_dim + 128, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_cell_states)
        )

    def forward(self, tokens, cell_features):
        # get transformer embeddings from BioBertModel
        outputs = self.base_model(tokens, return_cell_states=False)
        last_layer = max(outputs["representations"].keys())
        seq_repr = outputs["representations"][last_layer].mean(dim=1)  # mean-pooled sequence embeddings

        # encode cell-level numeric features
        cell_repr = self.cell_feature_encoder(cell_features)

        # fuse them (concat)
        fused = torch.cat([seq_repr, cell_repr], dim=1)
        logits = self.fusion_head(fused)

        return logits

In [ ]:
# Mock args and alphabet like before
args = type('', (), {})()
args.embed_dim = 1280
args.ffn_embed_dim = 5120
args.attention_heads = 20
args.layers = 4
args.arch = "roberta_large"
args.max_positions = 512
args.final_bias = False
alphabet = type('', (), {
    "padding_idx":0, "mask_idx":1, "cls_idx":2, "eos_idx":3,
    "prepend_bos":False, "append_eos":False, "__len__":lambda s:30
})()

# Base model (simulating your BioBert)
base_model = BioBertModel(args, alphabet, num_cell_states=3)

# Multimodal model
model = MultiModalCellStateModel(base_model, cell_feature_dim=10, num_cell_states=3)

# Generate random data
batch_size = 4
seq_len = 256
tokens = torch.randint(0, 30, (batch_size, seq_len))       # random sequences
cell_features = torch.randn(batch_size, 10)                 # random numeric data

# Run the model
logits = model(tokens, cell_features)

# Convert logits to probabilities
probs = F.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)

print("=== Multimodal Model Output ===")
print("Logits:\n", logits)
print("Probabilities:\n", probs)
print("Predicted Cell States:", preds)

acc = accuracy_fn(logits, labels)
print("Accuracy:", acc)